In [49]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [50]:
%pip install pandas
%pip install pytest

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Paye calculator

In [51]:
from decimal import Decimal
import pandas as pd
from src.paye import Paye
from src.tax_context import TaxContext

tx = Paye('UG')

# input map: gross_monthly, residency, expected PAYE
inputs = [
    (0, "resident", 0), 
    (300_000, "resident", 0), 
    (335_000, "resident", 0),
    (350_000, "resident", 3_000), 
    (410_000, "resident", 15_000), 
    (450_000, "resident", 25_000),
    (485_000, "resident", 33_750), 
    (500_000, "resident", 38_250), 
    (1_000_000, "resident", 188_250),
    (10_000_000, "resident", 2_888_250), 
    (12_000_000, "resident", 3_688_250),
    (200_000, "non_resident", 20_000), 
    (400_000, "non_resident", 46_500),
    (1_000_000, "non_resident", 225_500), 
    (12_000_000, "non_resident", 3_725_500),
]

rows = []
for gross, residency, expected in inputs:
    result = tx.calculate(TaxContext(gross_monthly=Decimal(gross), residency=residency))
    paye = int(result.amount)
    rows.append({
        "Gross (UGX)": gross,
        "Residency": residency.replace("_", "-").title(),
        "PAYE (UGX)": paye,
        "Expected (UGX)": expected,
        "Net pay (UGX)": gross - paye,
    })

df = pd.DataFrame(rows)

money_cols = ["Gross (UGX)", "PAYE (UGX)", "Expected (UGX)", "Net pay (UGX)"]
styled = (
    df.style
    .format("{:,.0f}", subset=money_cols)
    .hide(axis="index")
    .set_caption("Uganda PAYE test results (monthly)")
    .set_table_styles([
        {"selector": "caption", "props": "caption-side: top; font-size: 1.1em; font-weight: 600; text-align: left; padding: 6px 0;"},
        {"selector": "th", "props": "background-color: #1f4e79; color: white; text-align: right; padding: 6px 10px;"},
        {"selector": "td", "props": "text-align: right; padding: 4px 10px; border-bottom: 1px solid #e1e4e8;"},
        {"selector": "", "props": "border-collapse: collapse; font-family: sans-serif; font-size: 0.9em;"},
    ])
)

styled

Gross (UGX),Residency,PAYE (UGX),Expected (UGX),Net pay (UGX)
0,Resident,0,0,0
"300,000",Resident,0,0,"300,000"
"335,000",Resident,0,0,"335,000"
"350,000",Resident,"3,000","3,000","347,000"
"410,000",Resident,"15,000","15,000","395,000"
"450,000",Resident,"25,000","25,000","425,000"
"485,000",Resident,"33,750","33,750","451,250"
"500,000",Resident,"38,250","38,250","461,750"
"1,000,000",Resident,"188,250","188,250","811,750"
"10,000,000",Resident,"2,888,250","2,888,250","7,111,750"


## Payroll Pipeline

In [52]:
from dataclasses import replace
from decimal import Decimal
import pandas as pd
from src.paye import Paye
from src.nssf import Nssf
from src.lst import Lst
from src.tax_context import TaxContext

# ---- Settings ---------------------------------------------------------------
COUNTRY = "UG"
lst_enabled = True      # Local Service Tax on or off
pay_roll_month = 7       # LST is only deducted July-October

paye_tax, nssf_tax, lst_tax = Paye('UG'), Nssf('UG'), Lst('UG')

tax_types = paye_tax.tax_configurations["tax_types"]
LST_PRE_TAX = tax_types["LST"]["pre_tax_for_paye"]   
nssf_pre_tax = tax_types["NSSF"]["pre_tax_for_paye"] 

# sample employees: gross_monthly, residency
employees = [
    (300_000, "resident"),
    (450_000, "resident"),
    (1_000_000, "resident"),
    (1_500_000, "resident"),
    (12_000_000, "resident"),
    (1_000_000, "non_resident"),
]

rows = []
for gross, residency in employees:
    ctx = TaxContext(gross_monthly=Decimal(gross), residency=residency, month=pay_roll_month)

    # get employee LST and NSSF, both on gross pay
    lst = lst_tax.calculate(ctx).amount if lst_enabled else Decimal("0")
    nssf = nssf_tax.calculate(ctx)

    # get PAYE on gross less the deductions marked pre_tax_for_paye
    pre_tax = (lst if LST_PRE_TAX else 0) + (nssf.amount if nssf_pre_tax else 0)
    paye_base = ctx.gross_monthly - pre_tax
    paye = paye_tax.calculate(replace(ctx, gross_monthly=paye_base)).amount

    # get net pay; employer NSSF is the employer's cost, not a deduction
    net_pay = ctx.gross_monthly - lst - paye - nssf.amount

    rows.append({
        "Gross (UGX)": ctx.gross_monthly,
        "Residency": residency.replace("_", "-").title(),
        "LST (UGX)": lst,
        "PAYE base (UGX)": paye_base,
        "PAYE (UGX)": paye,
        "NSSF employee (UGX)": nssf.amount,
        "Net pay (UGX)": net_pay,
        "NSSF employer (UGX)": nssf.employer_amount,
    })

df = pd.DataFrame(rows)

money_cols = [c for c in df.columns if "(UGX)" in c]
styled = (
    df.style
    .format("{:,.0f}", subset=money_cols)
    .hide(axis="index")
    .set_caption(f"Uganda payroll, month {pay_roll_month} (LST {'on' if lst_enabled else 'off'})")
    .set_table_styles([
        {"selector": "caption", "props": "caption-side: top; font-size: 1.1em; font-weight: 600; text-align: left; padding: 6px 0;"},
        {"selector": "th", "props": "background-color: #1f4e79; color: white; text-align: right; padding: 6px 10px;"},
        {"selector": "td", "props": "text-align: right; padding: 4px 10px; border-bottom: 1px solid #e1e4e8;"},
        {"selector": "", "props": "border-collapse: collapse; font-family: sans-serif; font-size: 0.9em;"},
    ])
)

styled

Gross (UGX),Residency,LST (UGX),PAYE base (UGX),PAYE (UGX),NSSF employee (UGX),Net pay (UGX),NSSF employer (UGX)
"300,000",Resident,"2,500","297,500",0,"15,000","282,500","30,000"
"450,000",Resident,"7,500","442,500","23,125","22,500","396,875","45,000"
"1,000,000",Resident,"22,500","977,500","181,500","50,000","746,000","100,000"
"1,500,000",Resident,"25,000","1,475,000","330,750","75,000","1,069,250","150,000"
"12,000,000",Resident,"25,000","11,975,000","3,678,250","600,000","7,696,750","1,200,000"
"1,000,000",Non-Resident,"22,500","977,500","218,750","50,000","708,750","100,000"


## Order of deductions

| Step | Deduction | Worked out from | Lowers the pay PAYE is charged on? |
|---|---|---|---|
| 1 | Local Service Tax (LST) | Gross pay | Yes |
| 2 | NSSF (5% employee, 10% employer) | Gross pay | No |
| 3 | PAYE | Gross pay minus LST | |
| 4 | Net pay | Gross pay minus LST, PAYE and the employee's NSSF | |

### Why this order

LST comes off pay before PAYE is worked out, so it has to be calculated first.
If PAYE went first, the employee would be taxed on money that has already gone to LST.

NSSF does not lower the pay PAYE is charged on, so moving it before or after PAYE
changes nothing. The employer's 10% is paid on top of salary, so it is not taken
out of net pay.

Both rules come from the `pre_tax_for_paye` setting in `configs/tax.json`.

### Example

An employee earning UGX 1,500,000 in July pays UGX 25,000 LST.
PAYE is then worked out on 1,475,000, which gives 330,750, and net pay is 1,069,250.
Working out PAYE on the full 1,500,000 would take 7,500 too much.

LST is only charged from July to October. In the other months there is nothing
to take off first, so the order makes no difference.

The rule that LST lowers PAYE comes from payroll guides (SunPro and AfroTools),
not from URA directly, so it still needs checking with URA.

## VAT module

Every invoice is either a sale or a purchase, and falls into one of three groups:

| Group | VAT | Example |
|---|---|---|
| Standard-rated | 18% | Most goods and services |
| Zero-rated | 0% | Exports |
| Exempt | None | Items the VAT Act lists as exempt |

Zero-rated and exempt invoices both carry no VAT, but they work differently.
After a zero-rated sale, the business can still claim back the VAT it paid on purchases.
After an exempt sale, it can't.

Output VAT is the VAT the business charges on its sales.
Input VAT is the VAT it pays on its purchases and claims back.
Net VAT is output VAT minus input VAT. If it's positive, the business pays it to URA.
If it's negative, the business gets a credit or a refund.

In [53]:
from decimal import Decimal
import pandas as pd
from src.vat import Vat
from src.tax_context import TaxContext

vat_tax = Vat("UG")

# Invoices for one VAT period. Amounts exclude VAT.
# type: "sale" (output VAT) or "purchase" (input VAT)
# category: "standard" (18%), "zero_rated" (0%) or "exempt" (no VAT)
invoices = [
    {"invoice": "S-001", "type": "sale",     "category": "standard",   "amount": 10_000_000},
    {"invoice": "S-002", "type": "sale",     "category": "standard",   "amount": 4_500_000},
    {"invoice": "S-003", "type": "sale",     "category": "zero_rated", "amount": 6_000_000},
    {"invoice": "S-004", "type": "sale",     "category": "exempt",     "amount": 2_000_000},
    {"invoice": "P-101", "type": "purchase", "category": "standard",   "amount": 5_000_000},
    {"invoice": "P-102", "type": "purchase", "category": "standard",   "amount": 1_200_000},
    {"invoice": "P-103", "type": "purchase", "category": "zero_rated", "amount": 800_000},
    {"invoice": "P-104", "type": "purchase", "category": "exempt",     "amount": 1_000_000},
]

# VAT on each invoice
rows = []
for inv in invoices:
    ctx = TaxContext(amount=Decimal(inv["amount"]), vat_category=inv["category"])
    vat = vat_tax.calculate(ctx).amount
    rows.append({
        "Invoice": inv["invoice"],
        "Type": inv["type"].title(),
        "Category": inv["category"].replace("_", "-").title(),
        "Amount (UGX)": ctx.amount,
        "VAT (UGX)": vat,
        "Total (UGX)": ctx.amount + vat,
    })

df = pd.DataFrame(rows)

# Output VAT is charged on sales; input VAT is paid on purchases and claimed back
output_vat = df.loc[df["Type"] == "Sale", "VAT (UGX)"].sum()
input_vat = df.loc[df["Type"] == "Purchase", "VAT (UGX)"].sum()
net_vat = output_vat - input_vat

summary = pd.DataFrame([
    {"Item": "Output VAT (on sales)", "UGX": output_vat},
    {"Item": "Input VAT (on purchases)", "UGX": input_vat},
    {"Item": "Net VAT payable" if net_vat >= 0 else "Net VAT refundable", "UGX": abs(net_vat)},
])

table_styles = [
    {"selector": "caption", "props": "caption-side: top; font-size: 1.1em; font-weight: 600; text-align: left; padding: 6px 0;"},
    {"selector": "th", "props": "background-color: #1f4e79; color: white; text-align: right; padding: 6px 10px;"},
    {"selector": "td", "props": "text-align: right; padding: 4px 10px; border-bottom: 1px solid #e1e4e8;"},
    {"selector": "", "props": "border-collapse: collapse; font-family: sans-serif; font-size: 0.9em;"},
]

money_cols = [c for c in df.columns if "(UGX)" in c]
invoice_table = (
    df.style
    .format("{:,.0f}", subset=money_cols)
    .hide(axis="index")
    .set_caption("Invoices for the period")
    .set_table_styles(table_styles)
)

summary_table = (
    summary.style
    .format("{:,.0f}", subset=["UGX"])
    .hide(axis="index")
    .set_caption("VAT return summary")
    .set_table_styles(table_styles)
)

display(invoice_table, summary_table)

Invoice,Type,Category,Amount (UGX),VAT (UGX),Total (UGX)
S-001,Sale,Standard,"10,000,000","1,800,000","11,800,000"
S-002,Sale,Standard,"4,500,000","810,000","5,310,000"
S-003,Sale,Zero-Rated,"6,000,000",0,"6,000,000"
S-004,Sale,Exempt,"2,000,000",0,"2,000,000"
P-101,Purchase,Standard,"5,000,000","900,000","5,900,000"
P-102,Purchase,Standard,"1,200,000","216,000","1,416,000"
P-103,Purchase,Zero-Rated,"800,000",0,"800,000"
P-104,Purchase,Exempt,"1,000,000",0,"1,000,000"


Item,UGX
Output VAT (on sales),"2,610,000"
Input VAT (on purchases),"1,116,000"
Net VAT payable,"1,494,000"


## Corporate income tax

A company pays tax on its chargeable income:

chargeable income = revenue − allowable deductions

Allowable deductions are costs of running the business, like salaries, rent and utilities.
Some costs can't be deducted, such as fines, income tax itself and personal spending.
Equipment is also left out here, because the law spreads its cost over several years instead.

The tax is 30% of chargeable income for a resident company. If costs are higher than revenue,
there's a loss and no tax to pay.

In [54]:
from decimal import Decimal
import pandas as pd
from src.cit import Cit
from src.tax_context import TaxContext

cit_tax = Cit("UG")

# One company's figures for the year (UGX)
revenue = [
    {"item": "Sales", "amount": 850_000_000},
    {"item": "Service income", "amount": 30_000_000},
]

# allowable: True if the law lets the company subtract it from revenue
expenses = [
    {"item": "Salaries and wages",            "amount": 320_000_000, "allowable": True},
    {"item": "Office rent",                   "amount": 60_000_000,  "allowable": True},
    {"item": "Utilities",                     "amount": 18_000_000,  "allowable": True},
    {"item": "Marketing",                     "amount": 25_000_000,  "allowable": True},
    {"item": "Interest on business loan",     "amount": 12_000_000,  "allowable": True},
    {"item": "Fines and penalties",           "amount": 3_000_000,   "allowable": False},
    {"item": "Income tax paid",               "amount": 40_000_000,  "allowable": False},
    {"item": "Director's personal spending",  "amount": 5_000_000,   "allowable": False},
    {"item": "New equipment (capital cost)",  "amount": 50_000_000,  "allowable": False},
]

# 1. Total revenue
total_revenue = sum(Decimal(r["amount"]) for r in revenue)

# 2. Allowable deductions only
allowable = sum(Decimal(e["amount"]) for e in expenses if e["allowable"])
not_allowable = sum(Decimal(e["amount"]) for e in expenses if not e["allowable"])

# 3. Chargeable income, then the CIT rate
chargeable_income = total_revenue - allowable
result = cit_tax.calculate(TaxContext(amount=chargeable_income))

lines = pd.DataFrame(
    [{"Item": r["item"], "Type": "Revenue", "Allowable": "", "Amount (UGX)": r["amount"]} for r in revenue]
    + [{"Item": e["item"], "Type": "Expense", "Allowable": "Yes" if e["allowable"] else "No",
        "Amount (UGX)": e["amount"]} for e in expenses]
)

summary = pd.DataFrame([
    {"Item": "Total revenue", "UGX": total_revenue},
    {"Item": "Less allowable deductions", "UGX": allowable},
    {"Item": "Chargeable income", "UGX": chargeable_income},
    {"Item": f"CIT at {cit_tax.rate:.0%}", "UGX": result.amount},
    {"Item": "Expenses not allowed (ignored above)", "UGX": not_allowable},
])

table_styles = [
    {"selector": "caption", "props": "caption-side: top; font-size: 1.1em; font-weight: 600; text-align: left; padding: 6px 0;"},
    {"selector": "th", "props": "background-color: #1f4e79; color: white; text-align: right; padding: 6px 10px;"},
    {"selector": "td", "props": "text-align: right; padding: 4px 10px; border-bottom: 1px solid #e1e4e8;"},
    {"selector": "", "props": "border-collapse: collapse; font-family: sans-serif; font-size: 0.9em;"},
]

lines_table = (
    lines.style
    .format("{:,.0f}", subset=["Amount (UGX)"])
    .hide(axis="index")
    .set_caption("Revenue and expenses for the year")
    .set_table_styles(table_styles)
)

summary_table = (
    summary.style
    .format("{:,.0f}", subset=["UGX"])
    .hide(axis="index")
    .set_caption("Corporate income tax")
    .set_table_styles(table_styles)
)

display(lines_table, summary_table)

Item,Type,Allowable,Amount (UGX)
Sales,Revenue,,"850,000,000"
Service income,Revenue,,"30,000,000"
Salaries and wages,Expense,Yes,"320,000,000"
Office rent,Expense,Yes,"60,000,000"
Utilities,Expense,Yes,"18,000,000"
Marketing,Expense,Yes,"25,000,000"
Interest on business loan,Expense,Yes,"12,000,000"
Fines and penalties,Expense,No,"3,000,000"
Income tax paid,Expense,No,"40,000,000"
Director's personal spending,Expense,No,"5,000,000"


Item,UGX
Total revenue,"880,000,000"
Less allowable deductions,"435,000,000"
Chargeable income,"445,000,000"
CIT at 30%,"133,500,000"
Expenses not allowed (ignored above),"98,000,000"
